# 03 - Baselines

Question: how well can simple, no-learning guesses predict rain tomorrow? Any real model has to beat these.

In [3]:
import pandas as pd 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('../data.raw/weatherAUS.csv', parse_dates=['Date'])
d = df.dropna(subset=['RainTomorrow']).copy()
d['y'] = (d['RainTomorrow'] == 'Yes').astype(int)

train_end = d['Date'].quantile(0.8)
val_end = d['Date'].quantile(0.9)
train = d[d['Date'] <= train_end]
val = d[(d['Date'] > train_end) & (d['Date'] <= val_end)]

In [6]:
def score(name, y_true, y_pred):
    return {
        'baseline': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
    }

always_no = pd.Series(0, index=val.index)
same_as_today = (val['RainToday'] == 'Yes').astype(int)

results = pd.DataFrame([
    score('always_no', val['y'], always_no),
    score('same as today', val['y'], same_as_today)
])

print(results.round(3))

        baseline  accuracy  precision  recall     f1
0      always_no     0.772      0.000   0.000  0.000
1  same as today     0.753      0.459   0.454  0.457


Finding: "Always no" scores 0.772 accuracy with zero recall. "Same as today" scores 0.753 accuracy and F1 0.457.

Why it matters: The majority class sets a high accuracy floor without predicting any rain, so accuracy alone can't show a model is useful.

Decision: A model must beat 0.772 accuracy and 0.457 F1 on validation to count as an improvement.


Limit: Both baselines give a hard yes/no, so they can't be compared on how well a model ranks days by rain probability.
